## Imports

In [1]:
import pandas as pd
import requests
import json
from pathlib import Path
from zipfile import ZipFile
api_key = "iuEO68Q3WH3VppVcdQkMIjb6OSnEJuh4aFkU9cK8"

/home/joe/anaconda3/envs/bic/lib/python3.6/site-packages/requests/__init__.py:104: RequestsDependencyWarning: urllib3 (1.26.18) or chardet (5.0.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  RequestsDependencyWarning)


## API

### Get Agency Codes 

In [2]:
url=f"https://api.usa.gov/crime/fbi/cde/agency/byStateAbbr/CO?API_KEY={api_key}"
response_API = requests.get(url
)
#print(response_API.status_code)
data = response_API.text
parse_json = json.loads(data)


In [3]:
agencies = {}
for agc in parse_json:
   # print(agc)
 #   print(agc['ori'],agc["agency_name"],agc["agency_id"],agc["county_name"])
    agencies[agc['ori']] = agc["agency_name"]
    
print(agencies)

{'CO0010000': "Adams County Sheriff's Office", 'CO0010100': 'Aurora Police Department', 'CO0010200': 'Brighton Police Department', 'CO0010300': 'Commerce City Police Department', 'CO0010400': 'Thornton Police Department', 'CO0010500': 'Westminster Police Department', 'CO0010600': 'Northglenn Police Department', 'CO0010800': 'Federal Heights Police Department', 'CO0011100': 'University of Colorado: Denver', 'CO0020000': "Alamosa County Sheriff's Office", 'CO0020100': 'Alamosa Police Department', 'CO0020200': 'Adams State University', 'CO0030000': "Arapahoe County Sheriff's Office", 'CO0030100': 'Englewood Police Department', 'CO0030200': 'Littleton Police Department', 'CO0030300': 'Sheridan Police Department', 'CO0030400': 'Glendale Police Department', 'CO0030500': 'Cherry Hills Village Police Department', 'CO0030600': 'Bow Mar Police Department', 'CO0030700': 'Greenwood Village Police Department', 'CO0030800': 'Columbine Valley Police Department', 'CO0030900': 'Arapahoe Community Colle

### Get Crimes by Agency

In [25]:
# ori = Orginating Agency Identifier

def decodeCrime(parse_json,ori):
    columns = ["data_year"]
    columns[1:] = [key for key in parse_json["keys"]]
    dd = {}
    for col in columns:
        dd[col] = []
    for val in parse_json["data"]:
#        print(ori,val["data_year"])
        for k,v in val.items():

            dd[k].append(v)
    df = pd.DataFrame(dd)
    df.insert(0,column="Agency",value=agencies[ori])
    df.insert(1,column="ori",value=ori)
    
    return df

nagc=0
for ori in agencies:
    nagc+=1
    print(f"{nagc} of {len(agencies)} -  {ori}")
    url=f"https://api.usa.gov/crime/fbi/cde/arrest/agency/{ori}/all?from=1970&to=2023&API_KEY={api_key}"
    response_API = requests.get(url)
    #print(response_API.status_code)
    data = response_API.text
    parse_json = json.loads(data)
#    print(parse_json)
    tmp = decodeCrime(parse_json,ori)
    if nagc > 1:
        crime = pd.concat([crime,tmp])
    else:
        crime=tmp
 
print("DONE ")

1 of 249 -  CO0010000
2 of 249 -  CO0010100
3 of 249 -  CO0010200
4 of 249 -  CO0010300
5 of 249 -  CO0010400
6 of 249 -  CO0010500
7 of 249 -  CO0010600
8 of 249 -  CO0010800
9 of 249 -  CO0011100
10 of 249 -  CO0020000
11 of 249 -  CO0020100
12 of 249 -  CO0020200
13 of 249 -  CO0030000
14 of 249 -  CO0030100
15 of 249 -  CO0030200
16 of 249 -  CO0030300
17 of 249 -  CO0030400
18 of 249 -  CO0030500
19 of 249 -  CO0030600
20 of 249 -  CO0030700
21 of 249 -  CO0030800
22 of 249 -  CO0030900
23 of 249 -  CO0031100
24 of 249 -  CO0040000
25 of 249 -  CO0040100
26 of 249 -  CO0050000
27 of 249 -  CO0050100
28 of 249 -  CO0050700
29 of 249 -  CO0060000
30 of 249 -  CO0070000
31 of 249 -  CO0070100
32 of 249 -  CO0070300
33 of 249 -  CO0070400
34 of 249 -  CO0070500
35 of 249 -  CO0070600
36 of 249 -  CO0070800
37 of 249 -  CO0080000
38 of 249 -  CO0080100
39 of 249 -  CO0080200
40 of 249 -  CO0090000
41 of 249 -  CO0100000
42 of 249 -  CO0100100
43 of 249 -  CO0100200
44 of 249 -  CO01003

In [26]:
crime.shape

(9685, 34)

In [27]:
crime['data_year'].value_counts().sort_index()

1970.0     71
1971.0     70
1972.0     76
1973.0     69
1974.0    102
1975.0    108
1976.0    129
1977.0    156
1978.0    172
1979.0    173
1980.0    173
1981.0    182
1982.0    176
1983.0    186
1984.0    193
1985.0    195
1986.0    193
1987.0    199
1988.0    198
1989.0    194
1990.0    189
1991.0    199
1992.0    196
1993.0    190
1994.0    190
1995.0    190
1996.0    178
1997.0    194
1998.0    188
1999.0    196
2000.0    182
2001.0    176
2002.0    173
2003.0    180
2004.0    185
2005.0    193
2006.0    208
2007.0    201
2008.0    205
2009.0    210
2010.0    206
2011.0    210
2012.0    214
2013.0    215
2014.0    220
2015.0    224
2016.0    218
2017.0    218
2018.0    221
2019.0    222
2020.0    225
2021.0    233
2022.0    221
Name: data_year, dtype: int64

In [32]:
crime.rename(columns={"data_year":"year"},inplace=True)

In [33]:
columns = crime.columns


In [34]:
for col in columns[2:]:
    crime[col] = crime[col].astype(int)

In [35]:
crime.dtypes

Agency                                                                  object
ori                                                                     object
year                                                                     int64
Aggravated Assault                                                       int64
All Other Offenses (Except Traffic)                                      int64
Arson                                                                    int64
Burglary                                                                 int64
Curfew and Loitering Law Violations                                      int64
Disorderly Conduct                                                       int64
Driving Under the Influence                                              int64
Drug Abuse Violations - Grand Total                                      int64
Drunkenness                                                              int64
Embezzlement                                        

In [36]:
columnMap = {
 'Aggravated Assault':'aggravatedAssault',
 'All Other Offenses (Except Traffic)':'allOtherOffensesExceptTraffic',
 'Arson': 'arson',
 'Burglary':'burglary',
 'Curfew and Loitering Law Violations': 'curfewLoiteringLawViolations',
 'Disorderly Conduct':'disorderlyConduct',
 'Driving Under the Influence':'drivingUnderTheInfluence',
 'Drug Abuse Violations - Grand Total':'drugAbuseViolationsGrandTotal',
 'Drunkenness':'drunkenness',
 'Embezzlement':'embezzlement',
 'Forgery and Counterfeiting':'forgeryAndCounterfeiting',
 'Fraud':'fraud',
 'Gambling - Total':'gamblingTotal',
 'Human Trafficking - Commercial Sex Acts':'humanTraffickingCommercialSexActs',
 'Human Trafficking - Involuntary Servitude':'humanTraffickingInvoluntaryServitude',
 'Larceny - Theft':'larcenyTheft',
 'Liquor Laws':'liquorLaws',
 'Manslaughter by Negligence':'manslaughterByNegligence',
 'Motor Vehicle Theft':'motorVehicleTheft',
 'Murder and Nonnegligent Manslaughter': 'murderAndNonnegligentManslaughter',
 'Offenses Against the Family and Children': 'offensesAgainstTheFamilyAndChildren',
 'Prostitution and Commercialized Vice': 'prostitutionAndCommercializedVice',
 'Rape':'rape',
 'Robbery':'robbery',
 'Sex Offenses (Except Rape, and Prostitution and Commercialized Vice)':'sexOffensesExceptRapeAndProstitutionAndCommercializedVice',
 'Simple Assault':'simpleAssault',
 'Stolen Property: Buying, Receiving, Possessing':'stolenPropertyBuyingReceivingPossessing',
 'Suspicion':'suspicion',
 'Vagrancy':'vagrancy',
 'Vandalism':'vandalism',
 'Weapons: Carrying, Possessing, Etc.':'weaponsCarryingPossessingEtc'
}  

crime.rename(columns=columnMap,inplace=True)

In [37]:
crime.rename(columns={'Agency':'agency'},inplace=True)

In [38]:
for col in crime.columns:
    print(col)

agency
ori
year
aggravatedAssault
allOtherOffensesExceptTraffic
arson
burglary
curfewLoiteringLawViolations
disorderlyConduct
drivingUnderTheInfluence
drugAbuseViolationsGrandTotal
drunkenness
embezzlement
forgeryAndCounterfeiting
fraud
gamblingTotal
humanTraffickingCommercialSexActs
humanTraffickingInvoluntaryServitude
larcenyTheft
liquorLaws
manslaughterByNegligence
motorVehicleTheft
murderAndNonnegligentManslaughter
offensesAgainstTheFamilyAndChildren
prostitutionAndCommercializedVice
rape
robbery
sexOffensesExceptRapeAndProstitutionAndCommercializedVice
simpleAssault
stolenPropertyBuyingReceivingPossessing
suspicion
vagrancy
vandalism
weaponsCarryingPossessingEtc


In [39]:
crime.to_csv("crimeCO-1970-2022.csv",index=False)

In [40]:
crime.shape

(9685, 34)

In [17]:
crime.head()

,agency,ori,year,aggravatedAssault,allOtherOffensesExceptTraffic,arson,burglary,curfewLoiteringLawViolations,disorderlyConduct,drivingUnderTheInfluence,...,prostitutionAndCommercializedVice,rape,robbery,sexOffensesExceptRapeAndProstitutionAndCommercializedVice,simpleAssault,stolenPropertyBuyingReceivingPossessing,suspicion,vagrancy,vandalism,weaponsCarryingPossessingEtc
0,Adams County Sheriff's Office,CO0010000,1970,57,576,0,214,0,60,31,...,17,12,15,36,135,9,0,0,32,8
1,Adams County Sheriff's Office,CO0010000,1971,85,767,12,184,0,59,31,...,3,13,23,17,123,1,0,0,41,10
2,Adams County Sheriff's Office,CO0010000,1972,61,742,4,170,0,54,16,...,1,20,17,12,133,11,0,0,64,4
3,Adams County Sheriff's Office,CO0010000,1974,73,803,4,199,0,55,43,...,0,16,20,12,140,0,0,3,38,11
4,Adams County Sheriff's Office,CO0010000,1975,62,843,9,206,1,80,35,...,0,14,67,27,109,11,0,1,52,5


### Check Output File

In [41]:
df = pd.read_csv("crimeCO-1970-2022.csv")

In [43]:
df.shape

(9685, 34)

In [ ]:
df.columns

Index(['agency', 'ori', 'year', 'aggravatedAssault',
       'allOtherOffensesExceptTraffic', 'arson', 'burglary',
       'curfewLoiteringLawViolations', 'disorderlyConduct',
       'drivingUnderTheInfluence', 'drugAbuseViolationsGrandTotal',
       'drunkenness', 'embezzlement', 'forgeryAndCounterfeiting', 'fraud',
       'gamblingTotal', 'humanTraffickingCommercialSexActs',
       'humanTraffickingInvoluntaryServitude', 'larcenyTheft', 'liquorLaws',
       'manslaughterByNegligence', 'motorVehicleTheft',
       'murderAndNonnegligentManslaughter',
       'offensesAgainstTheFamilyAndChildren',
       'prostitutionAndCommercializedVice', 'rape', 'robbery',
       'sexOffensesExceptRapeAndProstitutionAndCommercializedVice',
       'simpleAssault', 'stolenPropertyBuyingReceivingPossessing', 'suspicion',
       'vagrancy', 'vandalism', 'weaponsCarryingPossessingEtc'],
      dtype='object')

### More Granularity

In [18]:

response_API = requests.get(
'https://api.usa.gov/crime/fbi/cde/nibrs/state/CO/all/offense/count?API_KEY=iuEO68Q3WH3VppVcdQkMIjb6OSnEJuh4aFkU9cK8')
#print(response_API.status_code)
data = response_API.text
parse_json = json.loads(data)

In [ ]:
parse_json

In [21]:
types = [
"count",
# "age",    
# "sex",    
# "race",    
# "ethnicity",    
# #"relationship",   on available for victim 
"location",    
"linkedoffense",    
"weapons"  
]

offenses = [
"all",    
"violent-crime",    
"aggravated-assault",
"burglary",    
"larceny",    
"motor-vehicle-theft",    
"homicide",    
"rape",    
"robbery",    
"arson",    
"property-crime",    

]

In [22]:

#url = f'https://api.usa.gov/crime/fbi/cde/nibrs/agency/CO0010000/{offen}/offender/{typ}?from=2016&to=2021&API_KEY==iuEO68Q3WH3VppVcdQkMIjb6OSnEJuh4aFkU9cK8'

In [ ]:
ori = "CO0010000"
dataAll = {}
dataAll[ori] = {}
# dataAll[ori]["data"] = []
# dataAll[ori]["what"] = []
cat = "offense"
fails=[]
for typ in types:
    for offen in offenses:
        try:
            url = f'https://api.usa.gov/crime/fbi/cde/nibrs/agency/{ori}/{offen}/{cat}/{typ}?from=2016&to=2021&API_KEY=iuEO68Q3WH3VppVcdQkMIjb6OSnEJuh4aFkU9cK8'        
          #  print(url)
            response_API = requests.get(url)
            print(ori,typ,offen,response_API)
            string = f"{cat},{typ},{offen}"
            data = response_API.text
            parse_json = json.loads(data)
            # dataAll[ori]["data"].append(parse_json)
            # dataAll[ori]["what"].append(string)
            if string not in dataAll[ori]:
                dataAll[ori][string] = []
            dataAll[ori][string].append(parse_json)
        except:
            print("FAIL ",string)
            fails.append(string)
        
print("DONE")

In [ ]:
dataAll['CO0010000']

In [34]:
dataAll['CO0010000']['data'][0]

{'keys': ['Incident Count', 'Offense Count'],
 'data': [{'data_year': 2017, 'Incident Count': 91, 'Offense Count': 100},
  {'data_year': 2019, 'Incident Count': 666, 'Offense Count': 666},
  {'data_year': 2020, 'Incident Count': 3, 'Offense Count': 3},
  {'data_year': 2020, 'Incident Count': 66, 'Offense Count': 72},
  {'data_year': 2017, 'Incident Count': 8, 'Offense Count': 8},
  {'data_year': 2018, 'Incident Count': 793, 'Offense Count': 824},
  {'data_year': 2021, 'Incident Count': 4, 'Offense Count': 4},
  {'data_year': 2018, 'Incident Count': 100, 'Offense Count': 100},
  {'data_year': 2021, 'Incident Count': 429, 'Offense Count': 486},
  {'data_year': 2017, 'Incident Count': 36, 'Offense Count': 36},
  {'data_year': 2021, 'Incident Count': 536, 'Offense Count': 536},
  {'data_year': 2021, 'Incident Count': 44, 'Offense Count': 45},
  {'data_year': 2020, 'Incident Count': 18, 'Offense Count': 18},
  {'data_year': 2017, 'Incident Count': 113, 'Offense Count': 113},
  {'data_year':

In [33]:
print(dataAll['CO0010000']['what'][1])
print(type(dataAll['CO0010000']['data'][1]))
for val in dataAll['CO0010000']['data'][1]['title']:
    print(val)

offense,count,violent-crime
<class 'dict'>


In [100]:
response_API = requests.get('https://api.usa.gov/crime/fbi/cde/nibrs/agency/CO0010000/all/offender/count?from=2016&to=2021&API_KEY=iuEO68Q3WH3VppVcdQkMIjb6OSnEJuh4aFkU9cK8')
data = response_API.text
parse_json = json.loads(data)
print(parse_json)

{'keys': ['Count'], 'data': [{'data_year': 2016, 'Count': 301}, {'data_year': 2016, 'Count': 551}, {'data_year': 2016, 'Count': 34}, {'data_year': 2016, 'Count': 12}, {'data_year': 2016, 'Count': 1}, {'data_year': 2016, 'Count': 468}, {'data_year': 2016, 'Count': 153}, {'data_year': 2016, 'Count': 127}, {'data_year': 2016, 'Count': 1132}, {'data_year': 2016, 'Count': 881}, {'data_year': 2016, 'Count': 921}, {'data_year': 2016, 'Count': 2}, {'data_year': 2016, 'Count': 10}, {'data_year': 2016, 'Count': 15}, {'data_year': 2016, 'Count': 33}, {'data_year': 2016, 'Count': 3}, {'data_year': 2016, 'Count': 305}, {'data_year': 2016, 'Count': 4}, {'data_year': 2016, 'Count': 50}, {'data_year': 2016, 'Count': 73}, {'data_year': 2016, 'Count': 1035}, {'data_year': 2016, 'Count': 16}, {'data_year': 2016, 'Count': 3}, {'data_year': 2016, 'Count': 8}, {'data_year': 2016, 'Count': 41}, {'data_year': 2016, 'Count': 6}, {'data_year': 2016, 'Count': 87}, {'data_year': 2016, 'Count': 133}, {'data_year':

In [12]:
parse_json

{'message': 'Missing Authentication Token'}

In [25]:
def seeCols(df):
    for col in sorted(df.columns):
        print(col)

# NIBRS

## Read Data

In [18]:
col = "incident_id"
off = pd.read_csv("data/NIBRS_OFFENSE.csv")
icd = pd.read_csv("data/NIBRS_incident.csv")

In [19]:
off.columns

Index(['data_year', 'offense_id', 'incident_id', 'offense_code',
       'attempt_complete_flag', 'location_id', 'num_premises_entered',
       'method_entry_code'],
      dtype='object')

In [20]:
print(off.shape)
print(icd.shape)

(382636, 8)
(327646, 15)


## Merge Offenses and Incidents

In [21]:
off = off.merge(icd,on=col,how="left")

In [22]:
off.shape

(382636, 22)

In [23]:
off[col].nunique()

327646

In [26]:
seeCols(off)

agency_id
attempt_complete_flag
cargo_theft_flag
cleared_except_date
cleared_except_id
data_home
data_year_x
data_year_y
did
incident_date
incident_hour
incident_id
incident_status
location_id
method_entry_code
nibrs_month_id
num_premises_entered
offense_code
offense_id
orig_format
report_date_flag
submission_date


In [38]:
#off["OFFENSE_TYPE_ID".lower()].value_counts()
len(off["OFFENSE_code".lower()].value_counts())

51

In [30]:
#off["OFFENSE_TYPE_ID"].isna().sum()
off["OFFENSE_CODE".lower()].isna().sum()

0

## Get Name of Offense Committed

In [31]:
offType = pd.read_csv("data/NIBRS_OFFENSE_TYPE.csv")

In [35]:
offType.columns

Index(['offense_code', 'offense_name', 'crime_against', 'ct_flag', 'hc_flag',
       'hc_code', 'offense_category_name', 'offense_group'],
      dtype='object')

In [37]:
#off["OFFENSE_TYPE_ID"].nunique()
off["OFFENSE_CODE".lower()].nunique()

51

In [36]:
#offType["OFFENSE_TYPE_ID"].nunique()
offType["OFFENSE_CODE".lower()].nunique()

86

## Merge Offense Name

In [31]:
off = off.merge(offType,on="OFFENSE_TYPE_ID",how="left")

In [32]:
off.shape

(328469, 30)

In [33]:
seeCols(off)

AGENCY_ID
ATTEMPT_COMPLETE_FLAG
CARGO_THEFT_FLAG
CLEARED_EXCEPT_DATE
CLEARED_EXCEPT_ID
CRIME_AGAINST
CT_FLAG
DATA_HOME
DATA_YEAR_x
DATA_YEAR_y
DID
HC_CODE
HC_FLAG
INCIDENT_DATE
INCIDENT_HOUR
INCIDENT_ID
INCIDENT_STATUS
LOCATION_ID
METHOD_ENTRY_CODE
NIBRS_MONTH_ID
NUM_PREMISES_ENTERED
OFFENSE_CATEGORY_NAME
OFFENSE_CODE
OFFENSE_GROUP
OFFENSE_ID
OFFENSE_NAME
OFFENSE_TYPE_ID
ORIG_FORMAT
REPORT_DATE_FLAG
SUBMISSION_DATE


In [34]:
off["OFFENSE_NAME"].nunique()

51

In [35]:
off["OFFENSE_NAME"].value_counts()

Destruction/Damage/Vandalism of Property       44538
All Other Larceny                              37098
Simple Assault                                 30330
Theft From Motor Vehicle                       29232
Burglary/Breaking & Entering                   22141
Shoplifting                                    22100
Motor Vehicle Theft                            21682
Drug/Narcotic Violations                       20837
Drug Equipment Violations                      12364
Theft From Building                            10915
Aggravated Assault                             10337
Theft of Motor Vehicle Parts or Accessories     8948
False Pretenses/Swindle/Confidence Game         6314
Credit Card/Automated Teller Machine Fraud      5827
Weapon Law Violations                           5773
Impersonation                                   5757
Counterfeiting/Forgery                          5592
Identity Theft                                  4220
Intimidation                                  

In [37]:
off['OFFENSE_CODE'].value_counts()

290    44538
23H    37098
13B    30330
23F    29232
220    22141
23C    22100
240    21682
35A    20837
35B    12364
23D    10915
13A    10337
23G     8948
26A     6314
26B     5827
520     5773
26C     5757
250     5592
26F     4220
13C     4033
120     3833
11A     2606
280     2468
11D     2265
100     1905
200     1048
11B      681
370      656
720      567
11C      528
26G      492
270      488
23E      473
40A      462
210      390
26E      314
23B      225
23A      221
09A      203
36B      144
40B      137
510      136
36A       69
64A       47
09C       17
09B       15
39B       11
26D       11
64B        7
39C        6
40C        4
39A        2
Name: OFFENSE_CODE, dtype: int64

In [47]:
off["AGENCY_ID"].nunique()

215

## Agencies

In [2]:
agc = pd.read_csv("data/agencies.csv")

In [44]:
agc.head()

,YEARLY_AGENCY_ID,AGENCY_ID,DATA_YEAR,ORI,LEGACY_ORI,COVERED_BY_LEGACY_ORI,DIRECT_CONTRIBUTOR_FLAG,DORMANT_FLAG,DORMANT_YEAR,REPORTING_TYPE,...,NIBRS_LEOKA_START_DATE,NIBRS_CT_START_DATE,NIBRS_MULTI_BIAS_START_DATE,NIBRS_OFF_ETH_START_DATE,COVERED_FLAG,COUNTY_NAME,MSA_NAME,PUBLISHABLE_FLAG,PARTICIPATED,NIBRS_PARTICIPATED
0,18262017,1826,2017,CO0010000,CO0010000,NaN,N,N,NaN,I,...,01-MAR-03,01-FEB-14,01-JAN-16,01-APR-13,N,ADAMS,"Denver-Aurora-Lakewood, CO",Y,Y,Y
1,18272017,1827,2017,CO0010100,CO0010100,NaN,N,N,NaN,I,...,01-MAR-03,01-FEB-14,01-JAN-16,01-APR-13,N,DOUGLAS; ADAMS; ARAPAHOE,"Denver-Aurora-Lakewood, CO",Y,Y,Y
2,18282017,1828,2017,CO0010200,CO0010200,NaN,N,N,NaN,I,...,01-JAN-06,01-FEB-14,01-JAN-16,01-APR-13,N,WELD; ADAMS,"Denver-Aurora-Lakewood, CO; Greeley, CO",Y,Y,Y
3,18292017,1829,2017,CO0010300,CO0010300,NaN,N,N,NaN,I,...,01-MAR-03,01-FEB-14,01-JAN-16,01-APR-13,N,ADAMS,"Denver-Aurora-Lakewood, CO",Y,Y,Y
4,18302017,1830,2017,CO0010400,CO0010400,NaN,N,N,NaN,I,...,01-SEP-12,01-JUL-14,01-FEB-16,01-APR-13,N,ADAMS,"Denver-Aurora-Lakewood, CO",Y,Y,Y


In [50]:
seeCols(agc)

0
0.1
AGENCY_ID
AGENCY_STATUS
AGENCY_TYPE_NAME
COUNTY_NAME
COVERED_BY_LEGACY_ORI
COVERED_FLAG
DATA_YEAR
DIRECT_CONTRIBUTOR_FLAG
DIVISION_CODE
DIVISION_NAME
DORMANT_FLAG
DORMANT_YEAR
FEMALE_CIVILIAN
FEMALE_OFFICER
LEGACY_ORI
MALE_CIVILIAN
MALE_OFFICER
MIP_FLAG
MSA_NAME
NCIC_AGENCY_NAME
NIBRS_CERT_DATE
NIBRS_CT_START_DATE
NIBRS_LEOKA_START_DATE
NIBRS_MULTI_BIAS_START_DATE
NIBRS_OFF_ETH_START_DATE
NIBRS_PARTICIPATED
NIBRS_START_DATE
ORI
PARENT_POP_GROUP_CODE
PARENT_POP_GROUP_DESC
PARTICIPATED
PED.FEMALE_CIVILIAN+PED.FEMALE_OFFICER
PED.MALE_OFFICER+PED.MALE_CIVILIAN
PE_REPORTED_FLAG
POPULATION
POPULATION_GROUP_CODE
POPULATION_GROUP_DESC
POPULATION_GROUP_ID
POP_SORT_ORDER
PUBLISHABLE_FLAG
PUB_AGENCY_NAME
PUB_AGENCY_UNIT
REGION_CODE
REGION_DESC
REGION_NAME
REPORTING_TYPE
SAI
STATE_ABBR
STATE_ID
STATE_NAME
STATE_POSTAL_ABBR
SUBMITTING_AGENCY_ID
SUBMITTING_AGENCY_NAME
SUBURBAN_AREA_FLAG
SUMMARY_RAPE_DEF
UCR_AGENCY_NAME
YEARLY_AGENCY_ID


In [63]:
#agc["NCIC_AGENCY_NAME"].value_counts()
#agc["MSA_NAME"].value_counts()
#agc["PUB_AGENCY_NAME"].value_counts()
#agc["DIVISION_NAME"].value_counts()
#agc["SUBMITTING_AGENCY_NAME"].value_counts()
#agc["SUBMITTING_AGENCY_NAME"].value_counts()
#agc["UCR_AGENCY_NAME"].value_counts()
#agc["SAI"].value_counts()
#agc["AGENCY_TYPE_NAME"].value_counts()
agc["0.1"].value_counts()



0    220
Name: 0.1, dtype: int64

In [65]:
for col in sorted(agc.columns):
    print(col)
    print(agc[col].value_counts())
    print("-------------------------------")

0
0    220
Name: 0, dtype: int64
-------------------------------
0.1
0    220
Name: 0.1, dtype: int64
-------------------------------
AGENCY_ID
2047    1
1832    1
1844    1
1843    1
1842    1
       ..
1940    1
1938    1
1937    1
1936    1
2050    1
Name: AGENCY_ID, Length: 220, dtype: int64
-------------------------------
AGENCY_STATUS
A    220
Name: AGENCY_STATUS, dtype: int64
-------------------------------
AGENCY_TYPE_NAME
City                     142
County                    58
University or College     14
Other                      3
Other State Agency         2
State Police               1
Name: AGENCY_TYPE_NAME, dtype: int64
-------------------------------
COUNTY_NAME
WELD                19
JEFFERSON           10
EL PASO              9
ARAPAHOE             9
GARFIELD             7
                    ..
JEFFERSON; ADAMS     1
SEDGWICK             1
PITKIN; EAGLE        1
HINSDALE             1
ADAMS; WELD          1
Name: COUNTY_NAME, Length: 74, dtype: int64
-------------

In [46]:
agc["AGENCY_ID"].nunique()

220

In [ ]:
dfFinal = off[[]]

## Merge Agency info

In [48]:
off = off.merge(agc,on="AGENCY_ID",how="left")

In [49]:
off.shape

(328469, 88)

## CIM Data

### Crime Offenses by Police District 2001-2016 in Colorado

In [ ]:
offCIM = pd.read_csv("https://data.colorado.gov/api/views/ya69-n6ta/rows.csv?accessType=DOWNLOAD")

In [74]:
offCIM.columns

Index(['year', 'policeDistrict', 'type', 'subType', 'count'], dtype='object')

In [75]:
offCIM["subType"].value_counts()

Attempted                   7815
Knife/Cutting Instrument    6898
Other Dangerous Weapon      6898
Hands/Feet/Fist             3449
Firearm                     3449
Unlawful Entry              3449
Other                       3449
By Firearm                  3449
Truck                       3449
Forced Entry                3449
Auto                        3449
StrongArm                   3449
Other Assaults              3449
By Force                     917
Name: subType, dtype: int64

In [13]:
len(df.columns)

NameError: name 'df' is not defined

In [55]:
offCIM["policeDistrict"].value_counts()

Montezuma County Sheriff      410
Pueblo PD                     410
Summit County Sheriff         387
Bent County Sheriff           364
Baca County Sheriff           364
                             ... 
Colorado Attorney General       3
Collbran PD                     2
Univ. Southern Colorado PD      2
Sanford PD                      2
Summit County Task Force        1
Name: policeDistrict, Length: 284, dtype: int64

In [42]:
offCIM["subType"].value_counts()

Attempted                   7815
Other Dangerous Weapon      6898
Knife/Cutting Instrument    6898
Auto                        3449
Truck                       3449
Other Assaults              3449
Forced Entry                3449
By Firearm                  3449
Other                       3449
Hands/Feet/Fist             3449
Firearm                     3449
Unlawful Entry              3449
StrongArm                   3449
By Force                     917
Name: subType, dtype: int64

### Crime Arrests by Police District 2001-2016 in Colorado

In [39]:
offArrCIM = pd.read_csv("https://data.colorado.gov/api/views/2e5i-5hfy/rows.csv?accessType=DOWNLOAD")

In [40]:
offArrCIM.head()

,year,adultCount,juvenileCount,type,policeDistrict
0,2015,0.0,0.0,Murder Non Negligent Manslaughter,7th Judicial District Drug Task Force
1,2015,0.0,0.0,Rape,7th Judicial District Drug Task Force
2,2015,0.0,0.0,Robbery,7th Judicial District Drug Task Force
3,2015,0.0,0.0,Aggravated Assault,7th Judicial District Drug Task Force
4,2015,0.0,0.0,Burglary,7th Judicial District Drug Task Force


In [41]:
for col in offArrCIM.columns:
    print(col)

year
adultCount
juvenileCount
type
policeDistrict


In [81]:
offArrCIM["policeDistrict"].value_counts()

Montezuma County Sheriff      486
Pueblo PD                     486
Summit County Sheriff         459
Lafayette PD                  432
Grand Junction PD             432
                             ... 
La Veta Marshal                 2
Collbran PD                     2
Sanford PD                      2
Univ. Southern Colorado PD      2
Summit County Task Force        1
Name: policeDistrict, Length: 284, dtype: int64

In [84]:
offArrCIM.loc[offArrCIM[["policeDistrict","type"]].str.contains("Adams County")]

,year,adultCount,juvenileCount,type,policeDistrict
594,2015,6.0,0.0,Murder Non Negligent Manslaughter,Adams County Sheriff
595,2015,11.0,2.0,Rape,Adams County Sheriff
596,2015,25.0,7.0,Robbery,Adams County Sheriff
597,2015,133.0,5.0,Aggravated Assault,Adams County Sheriff
598,2015,46.0,2.0,Burglary,Adams County Sheriff
...,...,...,...,...,...
92618,2016,36.0,0.0,Prostitution,Adams County Sheriff
92651,2016,1.0,0.0,Embezzlement,Adams County Sheriff
92870,2005,46.0,1.0,Other Family Offenses,Adams County Sheriff
92953,2016,3.0,NaN,Vagrancy,Adams County Sheriff


In [90]:
offArrCIM.groupby(["policeDistrict","type"]).count()

year  adultCount  \
policeDistrict                   type                                   
22nd Judical District Task Force Aggravated Assault     6           6   
                                 All Other Offenses     6           6   
                                 Arson                  6           6   
                                 Burglary               6           6   
                                 Curfew Violations      6           0   
...                                                   ...         ...   
Yuma PD                          Runaways              16           0   
                                 Stolen Property       16          16   
                                 Vagrancy              16          16   
                                 Vandalism             16          16   
                                 Weapons               16          16   

                                                     juvenileCount  
policeDistrict                   type                               
22nd Judical District Task Force Aggravated Assault              6  
                                 All Other Offenses              6  
                                 Arson                           6  
                                 Burglary                        6  
                                 Curfew Violations               6  
...                                                            ...  
Yuma PD                          Runaways                       16  
                                 Stolen Property                16  
                                 Vagrancy                        0  
                                 Vandalism                      16  
                                 Weapons                        16  

[7474 rows x 3 columns]

In [91]:
df.head()

,Agency,data_year,Aggravated Assault,All Other Offenses (Except Traffic),Arson,Burglary,Curfew and Loitering Law Violations,Disorderly Conduct,Driving Under the Influence,Drug Abuse Violations - Grand Total,...,Prostitution and Commercialized Vice,Rape,Robbery,"Sex Offenses (Except Rape, and Prostitution and Commercialized Vice)",Simple Assault,"Stolen Property: Buying, Receiving, Possessing",Suspicion,Vagrancy,Vandalism,"Weapons: Carrying, Possessing, Etc."
0,Adams County Sheriff's Office,2000,152,4557,9,49,6,63,607,212,...,1,9,10,14,386,37,0,0,107,32
1,Adams County Sheriff's Office,2001,127,4497,7,68,16,169,619,223,...,0,8,25,17,405,46,0,1,118,26
2,Adams County Sheriff's Office,2002,106,4629,6,66,11,132,538,251,...,2,5,10,31,404,87,0,0,99,21
3,Adams County Sheriff's Office,2003,107,5547,2,67,4,76,559,228,...,7,4,11,42,369,57,0,0,117,38
4,Adams County Sheriff's Office,2004,113,6019,4,71,12,108,500,328,...,4,4,19,43,350,68,0,0,103,38


In [82]:
df.head()

,Agency,data_year,Aggravated Assault,All Other Offenses (Except Traffic),Arson,Burglary,Curfew and Loitering Law Violations,Disorderly Conduct,Driving Under the Influence,Drug Abuse Violations - Grand Total,...,Prostitution and Commercialized Vice,Rape,Robbery,"Sex Offenses (Except Rape, and Prostitution and Commercialized Vice)",Simple Assault,"Stolen Property: Buying, Receiving, Possessing",Suspicion,Vagrancy,Vandalism,"Weapons: Carrying, Possessing, Etc."
0,Adams County Sheriff's Office,2000,152,4557,9,49,6,63,607,212,...,1,9,10,14,386,37,0,0,107,32
1,Adams County Sheriff's Office,2001,127,4497,7,68,16,169,619,223,...,0,8,25,17,405,46,0,1,118,26
2,Adams County Sheriff's Office,2002,106,4629,6,66,11,132,538,251,...,2,5,10,31,404,87,0,0,99,21
3,Adams County Sheriff's Office,2003,107,5547,2,67,4,76,559,228,...,7,4,11,42,369,57,0,0,117,38
4,Adams County Sheriff's Office,2004,113,6019,4,71,12,108,500,328,...,4,4,19,43,350,68,0,0,103,38


In [92]:
df.head()

,Agency,data_year,Aggravated Assault,All Other Offenses (Except Traffic),Arson,Burglary,Curfew and Loitering Law Violations,Disorderly Conduct,Driving Under the Influence,Drug Abuse Violations - Grand Total,...,Prostitution and Commercialized Vice,Rape,Robbery,"Sex Offenses (Except Rape, and Prostitution and Commercialized Vice)",Simple Assault,"Stolen Property: Buying, Receiving, Possessing",Suspicion,Vagrancy,Vandalism,"Weapons: Carrying, Possessing, Etc."
0,Adams County Sheriff's Office,2000,152,4557,9,49,6,63,607,212,...,1,9,10,14,386,37,0,0,107,32
1,Adams County Sheriff's Office,2001,127,4497,7,68,16,169,619,223,...,0,8,25,17,405,46,0,1,118,26
2,Adams County Sheriff's Office,2002,106,4629,6,66,11,132,538,251,...,2,5,10,31,404,87,0,0,99,21
3,Adams County Sheriff's Office,2003,107,5547,2,67,4,76,559,228,...,7,4,11,42,369,57,0,0,117,38
4,Adams County Sheriff's Office,2004,113,6019,4,71,12,108,500,328,...,4,4,19,43,350,68,0,0,103,38


In [43]:
df = pd.read_csv("https://data.colorado.gov/resource/xi5f-mkzt.csv")

In [44]:
df.columns

Index(['agency', 'ori', 'year', 'aggravatedassault',
       'allotheroffensesexcepttraffic', 'arson', 'burglary',
       'curfewloiteringlawviolations', 'disorderlyconduct',
       'drivingundertheinfluence', 'drugabuseviolationsgrandtotal',
       'drunkenness', 'embezzlement', 'forgeryandcounterfeiting', 'fraud',
       'gamblingtotal', 'humantraffickingcommerci', 'humantraffickinginvolunt',
       'larcenytheft', 'liquorlaws', 'manslaughterbynegligence',
       'motorvehicletheft', 'murderandnonnegligentman',
       'offensesagainstthefamily', 'prostitutionandcommercia', 'rape',
       'robbery', 'sexoffensesexceptrapeand', 'simpleassault',
       'stolenpropertybuyingrece', 'suspicion', 'vagrancy', 'vandalism',
       'weaponscarryingpossessingetc'],
      dtype='object')

In [6]:
agc[['ncic_agency_name','pub_agency_name','population','county_name']].head()

,ncic_agency_name,pub_agency_name,population
0,ADAMS,Adams,100212
1,AURORA,Aurora,392134
2,BRIGHTON,Brighton,41205
3,COMMERCE CITY,Commerce City,65817
4,THORNTON,Thornton,143055


In [4]:
agc.columns

Index(['yearly_agency_id', 'agency_id', 'data_year', 'ori', 'legacy_ori',
       'covered_by_legacy_ori', 'direct_contributor_flag', 'dormant_flag',
       'dormant_year', 'reporting_type', 'ucr_agency_name', 'ncic_agency_name',
       'pub_agency_name', 'pub_agency_unit', 'agency_status', 'state_id',
       'state_name', 'state_abbr', 'state_postal_abbr', 'division_code',
       'division_name', 'region_code', 'region_name', 'region_desc',
       'agency_type_name', 'population', 'submitting_agency_id', 'sai',
       'submitting_agency_name', 'suburban_area_flag', 'population_group_id',
       'population_group_code', 'population_group_desc',
       'parent_pop_group_code', 'parent_pop_group_desc', 'mip_flag',
       'pop_sort_order', 'summary_rape_def', 'pe_reported_flag',
       'male_officer', 'male_civilian', 'male_officer+male_civilian',
       'female_officer', 'female_civilian', 'female_officer+female_civilian',
       'officer_rate', 'employee_rate', 'nibrs_cert_date', 'nibrs_

In [11]:
agc
agc[["agency_id","ori",'ncic_agency_name','pub_agency_name','population','county_name',
    'agency_type_name']].to_csv("agency_short.csv",index=False)